In [ ]:
# =========================================================================
# STEP 1: FULL DATASET TRAINING & MODEL EXPORT (Run strictly in Google Colab)
# =========================================================================
import os
import torch
import kagglehub
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split
import copy

# ১. কাগল থেকে সম্পূর্ণ ৭,০২৩টি ইমেজের ডেটাসেট ডাউনলোড
print("Downloading FULL Kaggle dataset... Please wait.")
download_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
dataset_root = os.path.join(download_path, 'Training')

# ২. ইমেজ অগমেন্টেশন ও প্রিপসেসিং পাইপলাইন
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ৩. ফুল ডেটাসেট লোড এবং ৮০-২০ অনুপাতে স্প্লিট (Train & Validation)
full_data = datasets.ImageFolder(root=dataset_root, transform=data_transforms)
train_size = int(0.8 * len(full_data))
val_size = len(full_data) - train_size
train_data, val_data = random_split(full_data, [train_size, val_size])

# ৪. ডাটা লোডার তৈরি (Batch Size: 32)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

# ৫. GPU ডিভাইস অ্যাসাইনমেন্ট এবং ResNet-50 মডেল লোড
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training will run on: {device}")

resnet50 = models.resnet50(weights='DEFAULT')

# ব্যাকবোন প্যারামিটার ফ্রিজ করা (Transfer Learning)
for param in resnet50.parameters():
    param.requires_grad = False

# আমাদের ৪টি ক্লাসের জন্য কাস্টম ক্লাসিফায়ার হেড যুক্ত করা
resnet50.fc = nn.Sequential(
    nn.Linear(resnet50.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 4)
)
resnet50 = resnet50.to(device)

# ৬. লস ফাংশন, অপ্টিমাইজার এবং লার্নিং রেট শিডিউলার নির্ধারণ
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet50.fc.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# ৭. ৫০ ইপোকের ট্রেইনিং লুপ
num_epochs = 50
best_acc = 0.0
best_model_wts = copy.deepcopy(resnet50.state_dict())

print("\n" + "="*50)
print("Starting 50-Epoch Optimization Sequence on FULL Dataset...")
print("="*50 + "\n")

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    resnet50.train()
    running_loss, correct_train = 0.0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet50(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data)

    scheduler.step()
    epoch_loss = running_loss / train_size
    epoch_acc = correct_train.double() / train_size

    # --- VALIDATION PHASE ---
    resnet50.eval()
    val_loss, correct_val = 0.0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = resnet50(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / val_size
    epoch_val_acc = correct_val.double() / val_size

    print(f"Epoch {epoch+1:02d}/{num_epochs} | Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

    # বেস্ট মডেল ট্র্যাক এবং সেভ করা
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        best_model_wts = copy.deepcopy(resnet50.state_dict())

# ৮. বেস্ট ওয়েটস লোড করা এবং ফাইল হিসেবে এক্সপোর্ট করা
print("\n" + "="*50)
print(f"Training Complete! Best Validation Accuracy: {best_acc*100:.2f}%")
print("="*50)

resnet50.load_state_dict(best_model_wts)
torch.save(resnet50.state_dict(), 'resnet_best.pth')
print("\n[ SUCCESS ] Model weights successfully exported as 'resnet_best.pth'.")

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Training will run on: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 179MB/s]



Starting 50-Epoch Optimization Sequence on FULL Dataset...

Epoch 01/50 | Train Loss: 0.5849 Acc: 0.7908 | Val Loss: 0.3635 Acc: 0.8804
Epoch 02/50 | Train Loss: 0.3725 Acc: 0.8571 | Val Loss: 0.3141 Acc: 0.8821
Epoch 03/50 | Train Loss: 0.3250 Acc: 0.8783 | Val Loss: 0.2869 Acc: 0.9071
Epoch 04/50 | Train Loss: 0.2843 Acc: 0.8967 | Val Loss: 0.2584 Acc: 0.9107
Epoch 05/50 | Train Loss: 0.2743 Acc: 0.8940 | Val Loss: 0.2420 Acc: 0.9125
Epoch 06/50 | Train Loss: 0.2545 Acc: 0.9029 | Val Loss: 0.2499 Acc: 0.9152
Epoch 07/50 | Train Loss: 0.2417 Acc: 0.9080 | Val Loss: 0.2413 Acc: 0.9134
Epoch 08/50 | Train Loss: 0.2445 Acc: 0.9062 | Val Loss: 0.2251 Acc: 0.9187
Epoch 09/50 | Train Loss: 0.2407 Acc: 0.9094 | Val Loss: 0.2421 Acc: 0.9152
Epoch 10/50 | Train Loss: 0.2188 Acc: 0.9185 | Val Loss: 0.2286 Acc: 0.9152
Epoch 11/50 | Train Loss: 0.2186 Acc: 0.9125 | Val Loss: 0.2291 Acc: 0.9152
Epoch 12/50 | Train Loss: 0.2090 Acc: 0.9156 | Val Loss: 0.2302 Acc: 0.9152
Epoch 13/50 | Train Loss: 0